In [ ]:
# https://datalake.viettelcyber.com/gateway/ui/zeppelin/#/notebook/2MHJR3JCY

In [ ]:
%livy.pyspark

# src_db = "bitu_schema"
src_db = "bitu_silver_data"
tgt_db = "bitu_silver_clone"

base_location = "hdfs:///opt/datasets/hive/bitu_silver_clone"

tables = [
    "deals",
"deals_allocation",
"deals_allocation_test",
"kinhdoanh",
"payment_records",
"plan_fin",
"plan_spdv_kd",
"pricebook",
"revenue",
"sales_accounts",
# "trino_sql",
]

spark.sql("""
CREATE DATABASE IF NOT EXISTS {}
""".format(tgt_db))

for table in tables:
    print("Cloning table to HDFS (1 file): " + table)

    src_table = "{db}.{tb}".format(db=src_db, tb=table)
    tgt_table = "{db}.{tb}".format(db=tgt_db, tb=table)
    table_location = base_location + "/" + table

    # Read source
    df = spark.table(src_table)

    # Tạo bảng đích (external, chỉ rõ location)
    df.limit(0).createOrReplaceTempView("tmp_schema")

    spark.sql("""
    CREATE TABLE IF NOT EXISTS {tgt}
    USING PARQUET
    LOCATION '{loc}'
    AS
    SELECT * FROM {src_table}
    """.format(
        tgt=tgt_table,
        loc=table_location,
        src_table=src_table,
    ))
    spark.sql("""
INSERT OVERWRITE TABLE {tgt_table}
SELECT *
FROM {src_table}
""".format(tgt_table=tgt_table,src_table=src_table ))

    # Ghi data – gom thành 1 file
    # df.write \
    #   .mode("overwrite") \
    #   .format("parquet") \
    #   .parquet(table_location)

    print("✓ Done: " + table)

In [ ]:
%livy.pyspark
df.show(10)